# Tahap 10: Fine-tuning IndoBERT untuk klasifikasi sentimen (Google Colab, GPU)

**Sebelum mulai**
1. Menu **Runtime → Change runtime type → GPU** (T4 sudah cukup).
2. Salin folder `data/splits/` dari komputer Anda ke Google Drive dengan struktur:
   `MyDrive/penelitian-ojol/data/splits/` berisi `train.csv`, `val.csv`, `test.csv`, `split.csv`, `split_info.json`.
3. Jalankan sel dari atas ke bawah. **Mulai dengan `UJI = True`** (sel konfigurasi) untuk uji cepat; setelah lancar, ubah ke `False`.

**Aturan yang dipatuhi:** pembagian data sama dengan model lain (md5 diverifikasi); penyeimbangan kelas hanya pada data latih;
epoch dipilih dengan data validasi (early stopping); strategi/konfigurasi dipilih dengan macro-F1 **validasi**;
data uji hanya dipakai satu kali pada konfigurasi terpilih.

In [ ]:
!pip -q install transformers scikit-learn pandas matplotlib

In [ ]:
import copy, hashlib, json, os, random, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("PyTorch", torch.__version__, "| transformers", transformers.__version__, "| perangkat:", DEVICE)
if DEVICE.type != "cuda":
    print("PERINGATAN: GPU tidak aktif. Pelatihan di CPU akan sangat lambat. "
          "Runtime -> Change runtime type -> GPU, lalu jalankan ulang dari atas.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===================== KONFIGURASI (ubah di sini) =====================
BASE = "/content/drive/MyDrive/penelitian-ojol"      # folder proyek di Google Drive
UJI = True                                           # True = uji cepat (data kecil, 1 epoch); ubah ke False untuk hasil sebenarnya

MODEL_NAME = "indobenchmark/indobert-base-p1"        # alternatif: "indolem/indobert-base-uncased"
KOLOM_TEKS = "teks_ringan"                           # teks dengan pra-pemrosesan ringan (tanda baca dipertahankan)
MAX_LEN = 128
BATCH_LATIH, BATCH_EVAL = 32, 128
EPOCH_MAKS, SABAR = 5, 2                             # SABAR = patience early stopping (berdasarkan macro-F1 validasi)
LR_LIST = [2e-5]                                     # contoh grid: [2e-5, 3e-5, 5e-5]
STRATEGI_LIST = ["bobot_kelas", "oversampling"]      # penanganan ketidakseimbangan (hanya data latih)
WEIGHT_DECAY, WARMUP = 0.01, 0.1
SEED = 42
SEED_TAMBAHAN = []                                   # mis. [43, 44]: ulangi konfigurasi terpilih untuk menilai variasi antar-seed
SIMPAN_MODEL = True

SPLIT_DIR = f"{BASE}/data/splits"
TAB_DIR, FIG_DIR = f"{BASE}/outputs/tables", f"{BASE}/outputs/figures"
PRED_DIR, MODEL_DIR = f"{BASE}/outputs/predictions", f"{BASE}/outputs/models"
for d_ in (TAB_DIR, FIG_DIR, PRED_DIR, MODEL_DIR):
    os.makedirs(d_, exist_ok=True)
SFX = "_uji" if UJI else ""
if UJI:
    EPOCH_MAKS, N_UJI_LATIH, N_UJI_EVAL = 1, 2000, 600
ID2LABEL = {0: "negatif", 1: "netral", 2: "positif"}
KELAS = [0, 1, 2]
NAMA_KELAS = [ID2LABEL[k] for k in KELAS]

In [ ]:
# ===================== DATA: muat dan verifikasi pembagian =====================
info = json.load(open(f"{SPLIT_DIR}/split_info.json", encoding="utf-8"))
md5 = hashlib.md5(open(f"{SPLIT_DIR}/split.csv", "rb").read()).hexdigest()
assert md5 == info["md5_split_csv"], "split.csv berubah (md5 tidak cocok): semua model harus memakai pembagian yang sama"
split = pd.read_csv(f"{SPLIT_DIR}/split.csv", dtype={"reviewId": str})
data = {}
for n in ("train", "val", "test"):
    data[n] = pd.read_csv(f"{SPLIT_DIR}/{n}.csv", keep_default_na=False,
                          dtype={"reviewId": str, KOLOM_TEKS: str}).reset_index(drop=True)
    assert set(data[n]["reviewId"]) == set(split.loc[split["split"] == n, "reviewId"]), f"{n}.csv tidak sesuai split.csv"
assert (data["train"]["label_final"] == data["train"]["label_id"].map(ID2LABEL)).all(), "pemetaan label_id tidak sesuai"
if UJI:
    data["train"] = data["train"].sample(n=min(N_UJI_LATIH, len(data["train"])), random_state=SEED).reset_index(drop=True)
    data["val"] = data["val"].sample(n=min(N_UJI_EVAL, len(data["val"])), random_state=SEED).reset_index(drop=True)
    data["test"] = data["test"].sample(n=min(N_UJI_EVAL, len(data["test"])), random_state=SEED).reset_index(drop=True)
    print("[MODE UJI] data dikecilkan; angka hasil tidak bermakna")
y = {n: data[n]["label_id"].astype(int).to_numpy() for n in data}
print(f"Pembagian diverifikasi (md5 {md5[:10]}...): latih {len(y['train']):,}, validasi {len(y['val']):,}, uji {len(y['test']):,}")
print("Distribusi latih:", {ID2LABEL[k]: int((y['train'] == k).sum()) for k in KELAS})

In [ ]:
# ===================== TOKENISASI dan panjang teks =====================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    print("AutoTokenizer gagal, mencoba BertTokenizer:", e)
    from transformers import BertTokenizer
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer huruf kecil (do_lower_case):", getattr(tokenizer, "do_lower_case", "tidak diketahui"))

penuh = tokenizer(data["train"][KOLOM_TEKS].tolist(), truncation=False, verbose=False)["input_ids"]
panjang = np.array([len(x) for x in penuh])
print(f"Panjang token (data latih): median {np.median(panjang):.0f}, p90 {np.percentile(panjang, 90):.0f}, "
      f"p95 {np.percentile(panjang, 95):.0f}, p99 {np.percentile(panjang, 99):.0f}, maks {panjang.max()}")
print(f"Ulasan yang terpotong pada MAX_LEN={MAX_LEN}: {100 * np.mean(panjang > MAX_LEN):.2f}%")

enc = {n: tokenizer(data[n][KOLOM_TEKS].tolist(), truncation=True, max_length=MAX_LEN)["input_ids"] for n in data}

In [ ]:
# ===================== FUNGSI BANTU =====================
class DataUlasan(Dataset):
    def __init__(self, ids, labels):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.ids[i], int(self.labels[i])

def kumpul(batch):
    maks = max(len(x) for x, _ in batch)
    ids = torch.full((len(batch), maks), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(batch), maks), dtype=torch.long)
    for i, (x, _) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x)
        mask[i, :len(x)] = 1
    return ids, mask, torch.tensor([b[1] for b in batch], dtype=torch.long)

def set_seed_semua(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def indeks_oversampling(labels, seed):
    # menggandakan contoh kelas minoritas (acak, dengan pengembalian) hingga sama dengan kelas terbesar; HANYA data latih
    rng = np.random.default_rng(seed)
    jumlah = np.bincount(labels, minlength=3)
    idx = []
    for k in KELAS:
        ik = np.flatnonzero(labels == k)
        tambah = rng.choice(ik, size=jumlah.max() - len(ik), replace=True) if len(ik) < jumlah.max() else np.array([], dtype=int)
        idx.append(np.concatenate([ik, tambah]))
    return np.concatenate(idx)

def bobot_kelas(labels):
    jumlah = np.bincount(labels, minlength=3)
    return torch.tensor(len(labels) / (3.0 * jumlah), dtype=torch.float32)       # 'balanced'

def hitung_metrik(yt, yp):
    pr, rc, f1, _ = precision_recall_fscore_support(yt, yp, labels=KELAS, zero_division=0)
    m = {"akurasi": accuracy_score(yt, yp), "akurasi_seimbang": balanced_accuracy_score(yt, yp),
         "presisi_macro": pr.mean(), "recall_macro": rc.mean(), "f1_macro": f1.mean(),
         "f1_weighted": f1_score(yt, yp, average="weighted", zero_division=0)}
    for i, nm in enumerate(NAMA_KELAS):
        m[f"presisi_{nm}"], m[f"recall_{nm}"], m[f"f1_{nm}"] = pr[i], rc[i], f1[i]
    return m

def f1_macro_fn(a, b):
    return f1_score(a, b, labels=KELAS, average="macro", zero_division=0)

def f1_netral_fn(a, b):
    return f1_score(a, b, labels=[1], average="macro", zero_division=0)

def bootstrap_ci(yt, yp, fungsi, n=1000):
    rng = np.random.default_rng(SEED)
    yt, yp = np.asarray(yt), np.asarray(yp)
    nilai = [fungsi(yt[i], yp[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(n))]
    return tuple(np.nanpercentile(nilai, [2.5, 97.5]))

def muat_loader(nama, shuffle=False, indeks=None, generator=None, batch=BATCH_EVAL):
    ds = DataUlasan(enc[nama], y[nama])
    if indeks is not None:
        ds = torch.utils.data.Subset(ds, indeks)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, collate_fn=kumpul, generator=generator)

@torch.no_grad()
def prediksi_logit(model, loader):
    model.eval()
    keluar = []
    for ids, mask, _ in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            lg = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).logits
        keluar.append(lg.float().cpu().numpy())
    return np.concatenate(keluar)

In [ ]:
# ===================== PELATIHAN SATU KONFIGURASI =====================
def latih(strategi, lr, seed, tag):
    set_seed_semua(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)
    g = torch.Generator()
    g.manual_seed(seed)
    ytr = y["train"]
    if strategi == "oversampling":
        loader = muat_loader("train", shuffle=True, indeks=indeks_oversampling(ytr, seed), generator=g, batch=BATCH_LATIH)
        w = None
    else:
        loader = muat_loader("train", shuffle=True, generator=g, batch=BATCH_LATIH)
        w = bobot_kelas(ytr).to(DEVICE) if strategi == "bobot_kelas" else None
    loss_fn = nn.CrossEntropyLoss(weight=w)
    val_loader = muat_loader("val")
    tanpa_decay = ("bias", "LayerNorm.weight")
    grup = [{"params": [p for n, p in model.named_parameters() if not any(t in n for t in tanpa_decay)], "weight_decay": WEIGHT_DECAY},
            {"params": [p for n, p in model.named_parameters() if any(t in n for t in tanpa_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(grup, lr=lr)
    total = len(loader) * EPOCH_MAKS
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=AMP)

    terbaik, riwayat, tunggu = {"f1": -1.0, "epoch": 0, "state": None}, [], 0
    for epoch in range(1, EPOCH_MAKS + 1):
        model.train()
        t0, jml_loss, n_batch = time.perf_counter(), 0.0, 0
        for ids, mask, yb in loader:
            ids, mask, yb = ids.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
                logits = model(input_ids=ids, attention_mask=mask).logits
            loss = loss_fn(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            jml_loss += loss.item(); n_batch += 1
        lg = prediksi_logit(model, val_loader)
        pv = lg.argmax(axis=1)
        f1 = f1_macro_fn(y["val"], pv)
        riwayat.append({"tag": tag, "epoch": epoch, "loss_latih": jml_loss / n_batch, "val_f1_macro": f1,
                        "val_akurasi": accuracy_score(y["val"], pv), "detik": round(time.perf_counter() - t0, 1)})
        print(f"  [{tag}] epoch {epoch}: loss latih {riwayat[-1]['loss_latih']:.4f} | validasi macro-F1 {f1:.4f} | "
              f"akurasi {riwayat[-1]['val_akurasi']:.4f} | {riwayat[-1]['detik']:.0f} dtk")
        if f1 > terbaik["f1"]:
            terbaik = {"f1": f1, "epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "pv": pv}
            tunggu = 0
        else:
            tunggu += 1
            if tunggu >= SABAR:
                print(f"  early stopping (tidak membaik {SABAR} epoch); epoch terbaik = {terbaik['epoch']}")
                break
    ckpt = f"/content/ckpt_{tag}.pt"
    torch.save(terbaik["state"], ckpt)
    del model, opt, terbaik["state"]
    torch.cuda.empty_cache()
    return {"tag": tag, "strategi": strategi, "lr": lr, "seed": seed, "epoch_terbaik": terbaik["epoch"],
            "val_f1_macro": terbaik["f1"], "ckpt": ckpt, "pv": terbaik["pv"], "riwayat": riwayat}

In [ ]:
# ===================== JALANKAN KONFIGURASI, PILIH DARI DATA VALIDASI =====================
runs = []
for strategi in STRATEGI_LIST:
    for lr in LR_LIST:
        tag = f"{strategi}_lr{lr:g}_s{SEED}"
        print(f"\n=== {tag} ===")
        runs.append(latih(strategi, lr, SEED, tag))
        pd.DataFrame([{k: v for k, v in r.items() if k not in ("pv", "riwayat")} for r in runs]) \
            .to_csv(f"{TAB_DIR}/indobert_perbandingan_konfigurasi{SFX}.csv", index=False)   # simpan bertahap

tabel = pd.DataFrame([{k: v for k, v in r.items() if k not in ("pv", "riwayat", "ckpt")} for r in runs])
pd.DataFrame([h for r in runs for h in r["riwayat"]]).to_csv(f"{TAB_DIR}/indobert_riwayat_epoch{SFX}.csv", index=False)
print("\n=== PERBANDINGAN KONFIGURASI (macro-F1 data validasi) ===")
print(tabel.round(4).to_string(index=False))
terpilih = max(runs, key=lambda r: r["val_f1_macro"])
print(f"\nKonfigurasi terpilih: {terpilih['tag']} (macro-F1 validasi {terpilih['val_f1_macro']:.4f}, epoch {terpilih['epoch_terbaik']})")

In [ ]:
# ===================== EVALUASI AKHIR (data uji dipakai SEKALI, hanya konfigurasi terpilih) =====================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.load_state_dict(torch.load(terpilih["ckpt"], map_location="cpu"))
model.to(DEVICE)
lg_val = prediksi_logit(model, muat_loader("val")); pv = lg_val.argmax(axis=1)
lg_te = prediksi_logit(model, muat_loader("test")); pt = lg_te.argmax(axis=1)

akhir = []
for nama, yt, yp in (("val", y["val"], pv), ("test", y["test"], pt)):
    pd.DataFrame({"reviewId": data[nama]["reviewId"], "y_true": yt, "y_pred": yp}).to_csv(f"{PRED_DIR}/indobert_{nama}{SFX}.csv", index=False)
    m = hitung_metrik(yt, yp)
    lo, hi = bootstrap_ci(yt, yp, f1_macro_fn)
    nlo, nhi = bootstrap_ci(yt, yp, f1_netral_fn)
    akhir.append({"model": "IndoBERT (fine-tuning)", "strategi": terpilih["strategi"], "split": nama, **m,
                  "f1_macro_ci_bawah": lo, "f1_macro_ci_atas": hi, "f1_netral_ci_bawah": nlo, "f1_netral_ci_atas": nhi})
    cm = confusion_matrix(yt, yp, labels=KELAS)
    pd.DataFrame(cm, index=NAMA_KELAS, columns=NAMA_KELAS).to_csv(f"{TAB_DIR}/indobert_matriks_{nama}{SFX}.csv")
    if nama == "test":
        persen = cm / cm.sum(axis=1, keepdims=True) * 100
        fig, ax = plt.subplots(figsize=(5, 4.3))
        ax.imshow(persen, cmap="Blues", vmin=0, vmax=100)
        ax.set_xticks(range(3), NAMA_KELAS); ax.set_yticks(range(3), NAMA_KELAS)
        ax.set_xlabel("Prediksi"); ax.set_ylabel("Label sebenarnya")
        ax.set_title(f"IndoBERT (data uji, strategi: {terpilih['strategi']})", fontsize=10)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{cm[i, j]}\n({persen[i, j]:.0f}%)", ha="center", va="center", fontsize=9,
                        color="white" if persen[i, j] > 50 else "black")
        ax.grid(False); fig.tight_layout(); fig.savefig(f"{FIG_DIR}/indobert_matriks_uji{SFX}.png", dpi=200); plt.show()
akhir = pd.DataFrame(akhir)
akhir.round(4).to_csv(f"{TAB_DIR}/indobert_hasil_akhir{SFX}.csv", index=False)
pd.set_option("display.width", 200)
print("=== HASIL AKHIR ===")
print(akhir[["model", "strategi", "split", "akurasi", "presisi_macro", "recall_macro", "f1_macro", "f1_negatif", "f1_netral", "f1_positif"]].round(4).to_string(index=False))
t = akhir[akhir["split"] == "test"].iloc[0]
print(f"\nData uji: macro-F1 {t['f1_macro']:.4f} (CI 95% {t['f1_macro_ci_bawah']:.4f} - {t['f1_macro_ci_atas']:.4f}) | "
      f"F1 netral {t['f1_netral']:.4f} (CI {t['f1_netral_ci_bawah']:.4f} - {t['f1_netral_ci_atas']:.4f})")

per_app = []
for a, g in data["test"].assign(y=y["test"], p=pt).groupby("app"):
    per_app.append({"app": a, "n": len(g), "n_netral": int((g["y"] == 1).sum()), "f1_macro": f1_macro_fn(g["y"], g["p"]),
                    "f1_netral": f1_netral_fn(g["y"], g["p"]), "akurasi": accuracy_score(g["y"], g["p"])})
per_app = pd.DataFrame(per_app)
per_app.round(4).to_csv(f"{TAB_DIR}/indobert_per_aplikasi_uji{SFX}.csv", index=False)
print("\nPer aplikasi (netral sangat sedikit; tafsirkan hati-hati):")
print(per_app.round(4).to_string(index=False))

In [ ]:
# ===================== KURVA BELAJAR, PARAMETER, DAN MODEL =====================
rw = pd.DataFrame(terpilih["riwayat"])
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(rw["epoch"], rw["loss_latih"], marker="o", label="loss latih")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss latih")
ax2 = ax.twinx()
ax2.plot(rw["epoch"], rw["val_f1_macro"], marker="s", color="#ff7f0e", label="macro-F1 validasi")
ax2.set_ylabel("Macro-F1 validasi")
ax.set_title(f"Kurva belajar: {terpilih['tag']}")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/indobert_kurva_belajar{SFX}.png", dpi=200); plt.show()

json.dump({"model": MODEL_NAME, "kolom_teks": KOLOM_TEKS, "max_len": MAX_LEN, "batch_latih": BATCH_LATIH, "lr_diuji": LR_LIST,
           "strategi_diuji": STRATEGI_LIST, "terpilih": terpilih["tag"], "epoch_terbaik": terpilih["epoch_terbaik"],
           "weight_decay": WEIGHT_DECAY, "warmup": WARMUP, "seed": SEED, "md5_split": md5, "uji_coba": UJI,
           "torch": torch.__version__, "transformers": transformers.__version__, "amp_fp16": AMP},
          open(f"{TAB_DIR}/indobert_parameter{SFX}.json", "w"), indent=2)
if SIMPAN_MODEL and not UJI:
    model.save_pretrained(f"{MODEL_DIR}/indobert_terbaik"); tokenizer.save_pretrained(f"{MODEL_DIR}/indobert_terbaik")
    print("Model disimpan di", f"{MODEL_DIR}/indobert_terbaik")
print("Selesai. Unduh/salin dari Drive: outputs/predictions/indobert_*.csv dan outputs/tables/indobert_*.csv ke folder proyek di komputer Anda.")

In [ ]:
# ===================== (OPSIONAL) VARIASI ANTAR-SEED untuk konfigurasi terpilih =====================
# Isi SEED_TAMBAHAN pada sel konfigurasi (mis. [43, 44]) lalu jalankan sel ini. Ini TIDAK mengubah pemilihan konfigurasi;
# hanya melaporkan kestabilan hasil (rata-rata dan simpangan baku macro-F1 uji).
if SEED_TAMBAHAN:
    baris = [{"seed": SEED, "val_f1_macro": terpilih["val_f1_macro"], "test_f1_macro": f1_macro_fn(y["test"], pt), "test_f1_netral": f1_netral_fn(y["test"], pt)}]
    for s in SEED_TAMBAHAN:
        r = latih(terpilih["strategi"], terpilih["lr"], s, f"{terpilih['strategi']}_lr{terpilih['lr']:g}_s{s}")
        m_ = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
        m_.load_state_dict(torch.load(r["ckpt"], map_location="cpu")); m_.to(DEVICE)
        p_ = prediksi_logit(m_, muat_loader("test")).argmax(axis=1)
        baris.append({"seed": s, "val_f1_macro": r["val_f1_macro"], "test_f1_macro": f1_macro_fn(y["test"], p_), "test_f1_netral": f1_netral_fn(y["test"], p_)})
        del m_
    st = pd.DataFrame(baris)
    st.round(4).to_csv(f"{TAB_DIR}/indobert_variasi_seed{SFX}.csv", index=False)
    print(st.round(4).to_string(index=False))
    print(f"Macro-F1 uji: rata-rata {st['test_f1_macro'].mean():.4f}, sd {st['test_f1_macro'].std(ddof=1):.4f}")
else:
    print("SEED_TAMBAHAN kosong: sel ini dilewati.")